# מעבדה 07 — החוק השני ומנועי חום

במעבדה הזו תבנו מנועי חום, תמדדו את נצילויותיהם, ותנסו — ותיכשלו — לתכנן מנוע שיכה את
חסם קרנו.

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | כמות קבועה של גז אידיאלי, $N$ חלקיקים עם $f$ דרגות חופש ריבועיות |
| **דינמיקה** | צעדים כמו-סטטיים המחוברים לכדי מסלול סגור במישור $P$–$V$ |
| **גבול** | בוכנה נטולת חיכוך; דופן המוחלפת בין דיאתרמית לאדיאבטית |
| **צבר** | אינו רלוונטי — זוהי תרמודינמיקה, דבר אינו סופר מיקרו-מצבים |
| **מוזנח** | חיכוך, מסת הבוכנה, אי-אידיאליות של הגז, דליפות דרך האדיאבטות, משך הצעד |
| **תקף כאשר** | הצעדים איטיים ביחס לזמן הרלקסציה; המאגרים גדולים דיים כדי לא לשנות טמפרטורה |
| **אופני כישלון** | פעולה בקצב סופי, מחדשים, חומר עבודה בקרבת התעבות |

כל הפיזיקה נמצאת ב-`thermolab.engines` — פתחו וקראו. דבר בקורס הזה אינו מוסתר בתוך מסגרת
תוכנה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import engines
from thermolab.validation import relative_error

T_HOT = 600.0   # K — the hot reservoir
T_COLD = 300.0  # K — the cold reservoir
N_PARTICLES = 1000
V_START = 1.0e-3  # m^3

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"Carnot bound between {T_HOT:.0f} K and {T_COLD:.0f} K: "
      f"{engines.carnot_efficiency(T_HOT, T_COLD):.6f}")

## חלק 1 — לבנות מנוע קרנו ולהתבונן בו

ארבעה צעדים: להתפשט במגע עם המאגר החם, להתפשט כשהחום מנותק, להידחס כנגד המאגר הקר, ולהידחס
שוב כשהחום מנותק. המסלול נסגר, והשטח שהוא כולא הוא העבודה המסופקת.

In [ ]:
cycle = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, expansion_ratio=2.5)

fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.4, 1]})
for stroke, colour in zip(cycle.strokes,
                          ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"], strict=True):
    path = stroke.process.quasistatic_path
    plane.plot(path.volumes * 1e3, path.pressures, lw=2.4, color=colour)
plane.set_xlabel("volume (L)")
plane.set_ylabel("pressure (Pa)")
plane.set_title("the cycle")

bars.bar(["heat in", "work out", "heat dumped"],
         [cycle.heat_absorbed, cycle.work_output, cycle.heat_rejected],
         color=["#dc2626", "#0f172a", "#2563eb"])
bars.set_ylabel("energy per cycle (J)")
bars.set_title("the ledger")
plt.tight_layout()
plt.show()

print(f"heat absorbed   Q_h = {cycle.heat_absorbed:.4e} J")
print(f"work delivered  W   = {cycle.work_output:.4e} J")
print(f"heat rejected   Q_c = {cycle.heat_rejected:.4e} J")
print(f"efficiency          = {cycle.efficiency:.6f}")
print(f"Carnot bound        = {cycle.carnot_bound:.6f}")

שימו לב שעמודת העבודה היא בדיוק מחצית מעמודת החום הנכנס, ושעמודת החום הנפלט היא המחצית
האחרת. המנוע *מושלם* — כל צעד שלו הפיך — והוא בכל זאת זורק מחצית ממה שבלע.

### לנבא

לפני הרצת התא הבא: אתם עומדים לבנות את אותו מנוע מגז אחר (חד-אטומי, דו-אטומי, רב-אטומי) ועם
יחסי התפשטות שונים. מי מן השינויים האלה משנה את הנצילות, ובאיזה כיוון?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
print(f"{'f':>3} {'ratio':>7} {'N':>8}   efficiency")
print("-" * 40)
for dof in (3, 5, 6):
    for ratio in (1.2, 2.5, 8.0):
        eta = engines.carnot_cycle(
            N_PARTICLES, T_HOT, T_COLD, V_START, ratio, degrees_of_freedom=dof
        ).efficiency
        print(f"{dof:>3} {ratio:>7.1f} {N_PARTICLES:>8}   {eta:.12f}")

# And over four decades of engine size, at fixed gas and shape:
for n in (10, 1000, 100_000):
    eta = engines.carnot_cycle(n, T_HOT, T_COLD, V_START, 2.5).efficiency
    print(f"{3:>3} {2.5:>7.1f} {n:>8}   {eta:.12f}")

שתים-עשרה ספרות, בכל פעם. הגז אינו משנה; הגודל אינו משנה; צורת המסלול אינה משנה. זהו משפט
קרנו, וכדאי לשהות עליו: ההוכחה שבדף המודול אינה פותחת את המנוע כלל, ולכן דבר מפנימיותו של
המנוע אינו יכול להופיע בתשובה.

החומים עצמם *אינם* זהים — מנוע גדול יותר מעביר אנרגיה רבה יותר באופן פרופורציוני. היחס
ביניהם הוא הקבוע.

## חלק 2 — לנסות להכות את החסם

כעת הניסוי המעניין. בנו מנועים באקראי: כל גודל, כל יחס התפשטות, כל איכות מגע תרמי. בדקו אם
מי מהם עולה על $1 - T_c/T_h$.

In [ ]:
efficiencies = []
for seed in range(200):
    engine = engines.random_two_reservoir_engine(np.random.default_rng(seed), T_HOT, T_COLD)
    efficiencies.append(engine.efficiency)

efficiencies = np.array(efficiencies)
bound = engines.carnot_efficiency(T_HOT, T_COLD)

plt.figure(figsize=(7, 4))
plt.hist(efficiencies, bins=30, color="#2563eb", alpha=0.75)
plt.axvline(bound, color="#dc2626", lw=2.2, label=f"Carnot bound = {bound:.3f}")
plt.xlabel("measured efficiency")
plt.ylabel("engines")
plt.legend()
plt.show()

print(f"engines built:            {efficiencies.size}")
print(f"best efficiency found:    {efficiencies.max():.6f}")
print(f"Carnot bound:             {bound:.6f}")
print(f"how many exceeded it:     {(efficiencies > bound).sum()}")

אף אחד מהם. נסו לשנות את טווח הזרעים, את טמפרטורות המאגרים, את הגז — הקיר אינו זז.

היזהרו במה שזה מראה. מאתיים מנועים המכבדים חסם **אינם** הוכחה לכך שאי אפשר להכותו; שום מדגם
סופי אינו יכול להיות כזה, וכל מנוע כאן נבנה בידי קוד שכבר מיישם את הפיזיקה נכון. מה שהסריקה
באמת היא, זה *מבחן הפרכה*: אם הייתם יכולים לגרום לאחד מהם לעלות על החסם, אז או שהספרייה
שגויה או שהגזירה שגויה. כישלון לשבור דבר שניסיתם בכל כוחכם לשבור הוא ראיה, לא הוכחה.

## חלק 3 — להריץ אותו לאחור

כל צעד של מחזור הפיך ניתן להיפוך. עשו זאת והמנוע הופך למקרר: עבודה נכנסת, חום יוצא מן הצד
הקר.

In [ ]:
fridge = engines.reversed_carnot_cycle(N_PARTICLES, 298.0, 275.0, V_START, 2.5)

print(f"work consumed per cycle   = {-fridge.work_output:.4e} J")
print(f"heat lifted from the cold = {fridge.heat_absorbed:.4e} J")
print(f"coefficient of performance = {fridge.coefficient_of_performance:.4f}")
print(f"closed form T_c/(T_h-T_c)  = {engines.cop_refrigerator(298.0, 275.0):.4f}")
print()
print(f"as a heat pump, COP        = {engines.cop_heat_pump(298.0, 275.0):.4f}")
print(f"difference is exactly 1:     "
      f"{engines.cop_heat_pump(298.0, 275.0) - engines.cop_refrigerator(298.0, 275.0):.10f}")

# Asking a refrigerator for an efficiency is a category error, and the library says so.
try:
    _ = fridge.efficiency  # bound only so the access is not a bare expression
except ValueError as error:
    print(f"\nasking for its efficiency: {error}")

### האם המקרר שובר את החוק השני?

הוא אכן מוריד את האנטרופיה של המזון שבתוכו. הריצו את החשבונאות וראו מניין בא הפיצוי — זהו
הניסוי ההורג את הקריאה ש"מקרר מפר את החוק השני", משום ששתי האנטרופיות מחושבות ולא נטענות.

In [ ]:
T_COLD_IN, T_KITCHEN = 275.0, 298.0
heat_lifted = 1000.0  # J removed from the food

for label, cop in [("a real fridge", 3.2), ("the best possible fridge",
                                            engines.cop_refrigerator(T_KITCHEN, T_COLD_IN))]:
    work = heat_lifted / cop
    dumped = heat_lifted + work           # first law: everything lifted, plus the work
    food = -heat_lifted / T_COLD_IN       # the food's entropy really does fall
    kitchen = dumped / T_KITCHEN          # the kitchen's rises
    print(f"{label} (COP {cop:.2f}):")
    print(f"    food    dS = {food:+.4f} J/K")
    print(f"    kitchen dS = {kitchen:+.4f} J/K")
    total = food + kitchen
    # The reversible fridge sits exactly at zero, so what comes back there is rounding of
    # either sign. A bare "-0.0000" would read as precisely the violation this cell rules
    # out, so display it as the zero it is.
    shown = 0.0 if abs(total) < 1e-9 * abs(food) else total
    print(f"    total   dS = {shown:+.4f} J/K")
    assert food < 0.0                     # the misconception's premise is TRUE
    assert total > -1e-9 * abs(food)      # ...and its conclusion still does not follow
    print()

print("The premise holds and the conclusion fails: the second law constrains the total.")
print("Only the reversible fridge reaches zero, and nothing gets below it.")

מקדם ביצועים של כ-12 אינו הפרה של דבר. דבר אינו *מומר* כאן — אנרגיה **מועברת**, והעברת חום
במעלה עולה פחות מן הכמות המועברת. זו גם הסיבה לכך שמשאבת חום מחממת בית בשבריר ממה שעולה
מחמם התנגדותי.

## חלק 4 — לאן הולכת העבודה האבודה

מנוע ממשי אינו יכול לגעת במאגרים שלו בדיוק בטמפרטורות שלהם: חום אינו חוצה הפרש טמפרטורות אפס
בקצב סופי. תנו לגז פער בכל קצה והתבוננו בשני גדלים זזים יחד.

In [ ]:
gaps = np.linspace(0.0, 100.0, 26)
etas, produced = [], []
for gap in gaps:
    engine = engines.endoreversible_cycle(
        N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, hot_gap=float(gap), cold_gap=float(gap)
    )
    etas.append(engine.efficiency)
    produced.append(engine.entropy_produced)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(gaps, etas, lw=2.2, color="#2563eb")
left.axhline(engines.carnot_efficiency(T_HOT, T_COLD), ls="--", color="#dc2626",
             label="Carnot bound")
left.set_xlabel("temperature gap at each end (K)")
left.set_ylabel("efficiency")
left.legend()

right.plot(gaps, produced, lw=2.2, color="#d97706")
right.set_xlabel("temperature gap at each end (K)")
right.set_ylabel("entropy produced per cycle (J/K)")
plt.tight_layout()
plt.show()

הנצילות יורדת והאנטרופיה המיוצרת עולה, יחד. אין אלה שני אפקטים אלא אחד, וכדאי לבדוק את הקשר
המדויק ישירות — העבודה האבודה שווה לטמפרטורת המאגר הקר כפול האנטרופיה המיוצרת (תוצאת
גואי–סטודולה).

In [ ]:
engine = engines.endoreversible_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, 40.0, 40.0)
perfect = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5)

# Scale both to the same heat absorbed so the comparison is fair.
scaled_perfect_work = perfect.efficiency * engine.heat_absorbed
lost = scaled_perfect_work - engine.work_output

print(f"work delivered by the real engine   = {engine.work_output:.6e} J")
print(f"a perfect engine on the same heat   = {scaled_perfect_work:.6e} J")
print(f"work lost                           = {lost:.6e} J")
print(f"T_c * entropy produced              = {T_COLD * engine.entropy_produced:.6e} J")
print(f"relative difference                 = "
      f"{relative_error(lost, T_COLD * engine.entropy_produced):.2e}")

# --- the checks that also live in the project's test suite ---
scale = abs(cycle.strokes[0].process.start.internal_energy)

# 1. The loop closes: four independent closed forms must agree.
assert abs(cycle.internal_energy_drift) / scale < 1e-12

# 2. The first law closes around the loop.
assert abs(cycle.first_law_residual) / scale < 1e-12

# 3. A reversible cycle produces no entropy.
assert abs(perfect.entropy_produced) / perfect.entropy_scale < 1e-12

# 4. An irreversible one produces a strictly positive amount.
assert engine.entropy_produced / engine.entropy_scale > 1e-6

# 5. The area enclosed really is the work delivered (quadrature vs closed form).
assert relative_error(perfect.enclosed_area, perfect.work_output) < 1e-4

print("\nall five checks passed")

## חלק 5 — לחקור בעצמכם

המחוונים מאפשרים לכם לשנות את המאגרים ואת איכות המגע התרמי. שני ניסויים שכדאי לעשות:

1. קרבו את שתי טמפרטורות המאגרים זו לזו והתבוננו בנצילות מתמוטטת. זו הסיבה לכך שחום פסולת
   בדרגה נמוכה חסר ערך כמעט, ולא משנה כמה ממנו יש.
2. החזיקו את הטמפרטורות קבועות ופתחו את הפערים. התבוננו בכמה נצילות אתם מאבדים עבור הזכות
   לפעול בקצב סופי.

לחצו על **Run Interact** אחרי הזזת המחוונים — הפונקציה מציירת מחדש שני לוחות.

In [ ]:
import ipywidgets as widgets


def explore(t_hot=600.0, t_cold=300.0, gap=20.0, expansion_ratio=2.5):
    span = t_hot - t_cold
    gap = min(gap, 0.45 * span)  # keep the working temperatures from crossing
    engine = engines.endoreversible_cycle(
        N_PARTICLES, t_hot, t_cold, V_START, expansion_ratio,
        hot_gap=gap, cold_gap=gap,
    )
    bound = engines.carnot_efficiency(t_hot, t_cold)

    fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 3.8),
                                      gridspec_kw={"width_ratios": [1.4, 1]})
    for stroke, colour in zip(engine.strokes,
                              ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"],
                              strict=True):
        path = stroke.process.quasistatic_path
        plane.plot(path.volumes * 1e3, path.pressures, lw=2.2, color=colour)
    plane.set_xlabel("volume (L)")
    plane.set_ylabel("pressure (Pa)")

    bars.bar(["this engine", "Carnot bound"], [engine.efficiency, bound],
             color=["#2563eb", "#dc2626"])
    bars.set_ylim(0, 1)
    bars.set_ylabel("efficiency")
    plt.tight_layout()
    plt.show()

    print(f"efficiency        {engine.efficiency:.4f}")
    print(f"Carnot bound      {bound:.4f}")
    print(f"entropy produced  {engine.entropy_produced:.3e} J/K per cycle")


widgets.interact_manual(
    explore,
    t_hot=widgets.FloatSlider(min=350, max=1200, step=25, value=600),
    t_cold=widgets.FloatSlider(min=200, max=340, step=5, value=300),
    gap=widgets.FloatSlider(min=0, max=100, step=5, value=20),
    expansion_ratio=widgets.FloatSlider(min=1.2, max=8.0, step=0.2, value=2.5),
);

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "07-second-law.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. ניסיתם להכות את חסם קרנו ונכשלתם. הסבירו מדוע הכישלון הזה אינו הוכחה למשפט קרנו, ואמרו מה
   כן היה נחשב להוכחה.
3. עמית אומר שהחוק השני "עוסק בעצם רק בחיכוך ובאובדנים". תנו לו את התיקון במשפט אחד שמנוע
   מושלם ונטול חיכוך היה עדיין מכבד.

**התשובות שלכם:**

1.
2.
3.